In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

df = pd.read_excel("/content/business_regression_data.xlsx")  # adjust path to your repo
print(df.shape)
df.head()

(320, 14)


,store_id,month,region,store_type,marketing_spend,footfall,avg_discount_pct,staff_count,inventory_availability_pct,competitor_distance_km,holiday_flag,customer_rating,monthly_sales,monthly_profit
0,STR-1001,2025-01-01,East,Residential,70389.11,8302,0.042,17,75.0,5.28,1,3.1,658832.62,84349.93
1,STR-1001,2025-02-01,East,Residential,43827.89,8487,0.168,23,84.4,4.31,0,2.8,634436.00,106130.54
2,STR-1001,2025-03-01,East,Residential,59921.75,9113,0.133,20,88.0,12.00,0,3.2,678722.06,101883.86
3,STR-1001,2025-04-01,East,Residential,56856.63,9431,0.142,23,86.9,4.97,0,3.4,658920.30,97910.37
4,STR-1002,2025-01-01,East,High Street,48412.64,9545,0.231,21,97.6,6.94,1,4.2,741341.44,119208.17


In [3]:
print("Missing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate store-month combos:", df.duplicated(subset=['store_id', 'month']).sum())
print("\nregion values:", sorted(df['region'].unique()))
print("store_type values:", sorted(df['store_type'].unique()))
# Panel is 80 stores x 4 months = 320 rows, fully balanced, no duplicates.
# Only two columns have missing values, and both are small (<3% of rows).


Missing values:
 store_id                      0
month                         0
region                        0
store_type                    0
marketing_spend               0
footfall                      0
avg_discount_pct              0
staff_count                   0
inventory_availability_pct    0
competitor_distance_km        6
holiday_flag                  0
customer_rating               8
monthly_sales                 0
monthly_profit                0
dtype: int64

Duplicate rows: 0
Duplicate store-month combos: 0

region values: ['East', 'North', 'South', 'West']
store_type values: ['Airport', 'High Street', 'Mall', 'Residential']


In [4]:
# Median imputation: both columns are numeric with a handful of gaps, and
# median is robust to the outliers/skew that mean imputation would be
# sensitive to.
df['competitor_distance_km'] = df['competitor_distance_km'].fillna(
    df['competitor_distance_km'].median())
df['customer_rating'] = df['customer_rating'].fillna(
    df['customer_rating'].median())

# ---- CELL 4: Explore correlations with the target ----
num_cols = ['marketing_spend', 'footfall', 'avg_discount_pct', 'staff_count',
            'inventory_availability_pct', 'competitor_distance_km',
            'holiday_flag', 'customer_rating', 'monthly_sales', 'monthly_profit']
print(df[num_cols].corr()['monthly_sales'].sort_values(ascending=False))


monthly_sales                 1.000000
footfall                      0.858071
staff_count                   0.807645
monthly_profit                0.691222
marketing_spend               0.408956
holiday_flag                  0.195265
inventory_availability_pct    0.114521
customer_rating              -0.026428
avg_discount_pct             -0.091052
competitor_distance_km       -0.106975
Name: monthly_sales, dtype: float64


In [5]:
# region and store_type are categorical -> statsmodels' C() encodes them
# as dummy variables automatically (baseline category is dropped).
formula = ("monthly_sales ~ marketing_spend + footfall + avg_discount_pct + "
           "staff_count + inventory_availability_pct + competitor_distance_km + "
           "holiday_flag + customer_rating + C(region) + C(store_type)")
model = smf.ols(formula, data=df).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:          monthly_sales   R-squared:                       0.857
Model:                            OLS   Adj. R-squared:                  0.850
Method:                 Least Squares   F-statistic:                     130.5
Date:                Sat, 04 Jul 2026   Prob (F-statistic):          1.25e-119
Time:                        22:36:53   Log-Likelihood:                -3838.4
No. Observations:                 320   AIC:                             7707.
Df Residuals:                     305   BIC:                             7763.
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

In [6]:

X = pd.get_dummies(df[['marketing_spend', 'footfall', 'avg_discount_pct', 'staff_count',
                        'inventory_availability_pct', 'competitor_distance_km',
                        'holiday_flag', 'customer_rating', 'region', 'store_type']],
                    drop_first=True).astype(float)
X = sm.add_constant(X)
vif = pd.DataFrame({
    'variable': X.columns,
    'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})
print(vif[vif['variable'] != 'const'].sort_values('VIF', ascending=False))
# Rule of thumb: VIF > 10 signals a multicollinearity problem. Check the output -
# footfall and staff_count are correlated with each other but should stay well
# under that threshold.


                      variable       VIF
2                     footfall  6.561815
4                  staff_count  6.509579
12      store_type_High Street  3.407305
14      store_type_Residential  3.266979
13             store_type_Mall  2.893499
11                 region_West  1.416045
9                 region_North  1.370938
10                region_South  1.330425
1              marketing_spend  1.075504
7                 holiday_flag  1.061809
8              customer_rating  1.055594
3             avg_discount_pct  1.054256
6       competitor_distance_km  1.043424
5   inventory_availability_pct  1.032442


In [7]:
resid = model.resid
fitted = model.fittedvalues
print("Residual mean (should be ~0):", resid.mean())
print("Residual std:", resid.std())
# For a full write-up you could also plot fitted vs residuals here:
# import matplotlib.pyplot as plt
# plt.scatter(fitted, resid); plt.axhline(0, color='red'); plt.xlabel('Fitted'); plt.ylabel('Residuals')

Residual mean (should be ~0): 1.0506082617212087e-07
Residual std: 39243.3308119431


In [8]:
formula_profit = formula.replace("monthly_sales", "monthly_profit")
model_profit = smf.ols(formula_profit, data=df).fit()
print(model_profit.summary())

                            OLS Regression Results                            
Dep. Variable:         monthly_profit   R-squared:                       0.574
Model:                            OLS   Adj. R-squared:                  0.554
Method:                 Least Squares   F-statistic:                     29.32
Date:                Sat, 04 Jul 2026   Prob (F-statistic):           2.50e-48
Time:                        22:37:27   Log-Likelihood:                -3594.4
No. Observations:                 320   AIC:                             7219.
Df Residuals:                     305   BIC:                             7275.
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       